In [7]:
from sqlalchemy import create_engine
import pandas as pd
from queries import get_rentals_month_query

In [8]:
def rentals_month(connection ,month, year):
    query = get_rentals_month_query()
    df = pd.read_sql(query, connection, params=(month, year))
    return df

In [9]:
def rental_count_month(df):
    rental_counts = df.groupby('customer_id')['rental_id'].count().reset_index()
    rental_counts.columns = ['customer_id', 'rental_count']
    rental_counts = rental_counts.sort_values(by='rental_count', ascending=False)
    return rental_counts

In [10]:
def compare_rentals(df1,df2):
    merged_df = pd.merge(df1, df2, on='customer_id', how='inner', suffixes=('_m1', '_m2'))
    merged_df['difference'] = merged_df['rental_count_m2'] - merged_df['rental_count_m1']
    return merged_df

In [12]:
if __name__ == "__main__":
    try:
        USER = 'root'
        PASSWORD = '35459583'
        HOST = 'localhost'
        PORT = '3306'
        DATABASE = 'sakila'
        sql_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
        connection = create_engine(sql_string)
        df_rentals_Aug = rentals_month(connection, 8, 2005)
        df_rentals_Jul = rentals_month(connection, 6, 2005)
        if df_rentals_Jul is not None and df_rentals_Aug is not None:
            customer_rental_count_Aug = rental_count_month(df_rentals_Aug)
            customer_rental_count_Jul = rental_count_month(df_rentals_Jul)
            merged_df= compare_rentals(customer_rental_count_Jul,customer_rental_count_Aug)
            display(merged_df)
        else:
            print("No data found")
    except Exception as e:
        print(e)


,customer_id,rental_count_m1,rental_count_m2,difference
0,31,11,4,-7
1,454,10,11,1
2,329,9,10,1
3,295,9,9,0
4,561,9,8,-1
...,...,...,...,...
585,496,1,9,8
586,370,1,10,9
587,315,1,5,4
588,198,1,16,15
